In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Data Analytics: A2A SDK API Sample

This notebook demonstrates how to interact with the **DataA2Aservice** using the high-level A2A Python SDK.

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/a2a_sdk_sample.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fagents%2Fgemini_data_analytics%2Fa2a_sdk_sample.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/agents/gemini_data_analytics/a2a_sdk_sample.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/a2a_sdk_sample.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/generative-ai/logos/GitHub_Invertocat_Dark.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>


# Background and Overview
The **Conversational Analytics API** (also known as Gemini Data Analytics) lets you chat with your BigQuery or Looker data anywhere. This notebook demonstrates how to use the high-level usage patterns off standard developer operations.

This is a **Pre-GA** product. See [documentation](https://cloud.google.com/gemini/docs/conversational-analytics-api/overview) for more details.

Please provide feedback to conversational-analytics-api-feedback@google.com
<br>
### This notebook will help you
1. Authenticate to Google Cloud and Setup Environment
2. Install a2a-sdk Library
3. Instantiate a High-level A2A Client
4. Fulfill requests synchronously without needing async loops


In [ ]:
# @title 1. Environment Setup
# Install core dependencies
%pip install google-auth requests httpx nest_asyncio

import os
import time
import uuid
from google.auth import default
from google.auth.transport.requests import Request
from google.colab import auth

# Authenticate the user
auth.authenticate_user()

# Get credentials and project ID
creds, _ = default()
creds.refresh(Request())
access_token = creds.token

ENDPOINT = "https://geminidataanalytics.googleapis.com"
PROJECT_ID = "[your-project-id]"  # @param {type:"string"}
LOCATION = "global"  # @param {type:"string"}
# AGENT_ID can be found from the Cloud URL, e.g.
# https://console.cloud.google.com/bigquery/agents_hub/<your-agent-id>?project=<your-project-id>
AGENT_ID = "your-agent-id"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]"
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
if not LOCATION:
    LOCATION = os.environ.get("GOOGLE_CLOUD_REGION")
    
TENANT = f"projects/{PROJECT_ID}/locations/{LOCATION}/agents/{AGENT_ID}"

print(f"Target Tenant: {TENANT}")


## 2. Install A2A SDK

Pulling library mounts drawn from high level ADK frameworks instead of manual stubs!

In [ ]:
# @title Fetch and reference SDK
# Running external library installs natively on Sandbox environment
%pip install a2a-sdk

from a2a.client import Client as A2AClient
from a2a.types import AgentCard

print("High-level SDK imported successfully. Please restart runtime if import fails directly after execution!")

## 3. Implementation: SDK Native Client

Create constructor wrappers using authentic SDK client hooks operations of pure synchronous execution!

In [ ]:
import uuid
import asyncio
import httpx
import nest_asyncio
from a2a.client import Client as A2AClient
from a2a.client.client_factory import ClientFactory as A2AClientFactory
from a2a.client.client import ClientConfig as A2AClientConfig
from a2a.types import TransportProtocol as A2ATransport, AgentCapabilities
from a2a.types import AgentCard

nest_asyncio.apply()

class DataA2AClient:

  def __init__(self, endpoint, token):
    self.endpoint = f"https://{endpoint}" if not endpoint.startswith("http") else endpoint
    self.token = token
    httpx_client = httpx.AsyncClient(
        headers={"Authorization": f"Bearer {self.token}"},
        timeout=60.0
    )
    client_config = A2AClientConfig(
        streaming=True,
        polling=True,
        httpx_client=httpx_client,
        supported_transports=[A2ATransport.http_json]
    )
    factory = A2AClientFactory(config=client_config)
    card = AgentCard(
        url=f"{self.endpoint}/v1beta/a2a/{TENANT}/", 
        name="TargetAgent",
        description="Test Agent",
        version="1.0",
        preferred_transport="HTTP+JSON",
        capabilities=AgentCapabilities(),
        default_input_modes=[],
        default_output_modes=[],
        skills=[]
    )
    self.client = factory.create(card)

  async def send_message(self, tenant, text):
    from a2a.types import Message as A2AMessage
    
    message = A2AMessage(
        message_id=str(uuid.uuid4()),
        role="user",
        parts=[{"text": text}]
    )
    
    responses = self.client.send_message(message)
    async for response in responses:
      if hasattr(response, "status") and response.status:
        print(f"[Status] {response.status.state}")
      elif hasattr(response, "artifact") and response.artifact:
        print(f"[Artifact] {response.artifact.name}: {response.artifact.description}")
      elif hasattr(response, "parts") and response.parts:
        part = response.parts[0]
        if hasattr(part, "root") and part.root:
          print(f"[Message] {part.root.text}")
        elif hasattr(part, "text") and part.text:
          print(f"[Message] {part.text}")

client = DataA2AClient(ENDPOINT, access_token)
print("Client created successfully using authentic A2A SDK!")

## 4. Example: Standard Resolution

We can now receive targeted execution states naturally mapped on list loops!

In [ ]:
query = "Show me sales trends for 2025"
print(f"Sending request: {query}\n")

try:
  asyncio.run(client.send_message(TENANT, query))
except Exception as e:
  print(f"Error: {e}")

## 5. Cleanup

It is good practice to clean up any temporary resources or local session state.

In [ ]:
# @title Resource Cleanup
print(
    "No specific cloud resources were created in this demo that require manual"
    " deletion."
)
print(
    "However, you can use this section to reset any local session state if"
    " needed."
)